<a href="https://colab.research.google.com/github/hannahishimwe/colab_projects/blob/main/notebooks/rpgle_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports

(some may be unnecessary I used import list from LEX training modules)

In [ ]:
!pip install langchain
!pip install langchain-core
!pip install --upgrade langchain-community
!pip install pip-system-certs
!pip install python-certifi-win32
!pip install langchain-huggingface
!pip install huggingface_hub
!pip install langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [python-certifi-win32]


##Custom RPGLoader class

Hannah Ishimwe

This will allow a user to load RPG files as documents to be ingested by an LLM through Langchain.

Works variably with a whole repository or even just a single file without any extra input from user.

In [ ]:
import os
import re
from typing import List
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document

"""
Custom loader for RPGLE files in LangChain. Restricted to only RPGLE files.

Param: Path: a whole repository, or a single file.

Return: List of Documents, separating procedures from the rest of the file.

Metadata includes:
  - Filepath
  - Type
  - Format
  - Procedure name (if what was extracted was a procedure)
"""
class RPGLoader(BaseLoader):

    def __init__(self, path: str):
        self.path = path

        """
        Error checking in case user inputs a non-existent path.
        """
        if os.path.isdir(path):
            self.is_repository = True
        elif os.path.isfile(path):
            self.is_repository = False
        else:
            raise ValueError(f"Path '{path}' is neither a file nor a directory.")

    def load(self) -> List[Document]:
      """
      Main method used, selects correct method to create Document objects dependent on
      whether user is loading a repository or a single file.

      Params: None
      Returns: List of Documents
      """
      if self.is_repository:
        return self.load_repository()
      else:
        return self.load_document(self.path)


    def find_rpg_files(self, root_dir: str) -> List[str]:
      """
      Method to find all RPGLE files in a repository.

      Params: Filepath, str
      Returns: List of RPGLE files as strings
      """
      pattern = re.compile(r'\.rpgle', re.IGNORECASE)
      rpg_files = []
      for dirpath, _, files in os.walk(root_dir):
          for file in files:
              if pattern.search(file):
                  rpg_files.append(os.path.join(dirpath, file))
      return rpg_files


    def load_repository(self) -> List[Document]:
        """
        Method to apply load_document to all RPGLE files in a repository.

        Params: None
        Returns: List of Documents
        """
        rpg_files = self.find_rpg_files(self.path)
        all_docs = []
        for rpg_file in rpg_files:
            doc = self.load_document(rpg_file)
            all_docs.extend(doc)
        return all_docs

    def load_document(self, path: str) -> List[Document]:
      chosen_path = path
      with open(chosen_path, 'r', encoding='utf-8') as f:
          source = f.read()
          source = source.replace('\r\n', '\n').replace('\r', '\n') # normalise to prevent OS-related formatting issues when analysing

      format = "free" if source.strip().lower().startswith("**free") else "fixed"
      doc = Document(
                page_content=source,
                metadata={
                    "source": chosen_path,
                    "format": format
                }
            )

      return [doc]



###Testing

In [ ]:
!git clone https://github.com/sitemule/ILEastic.git

Cloning into 'ILEastic'...
remote: Enumerating objects: 1984, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 1984 (delta 141), reused 132 (delta 129), pack-reused 1818 (from 1)
Receiving objects: 100% (1984/1984), 746.20 KiB | 2.37 MiB/s, done.
Resolving deltas: 100% (1303/1303), done.


###Attempt with just one document first

Prompting LLM to be able to identify all procedure names.

In [ ]:
loader = RPGLoader("/content/ILEastic/unittests/mediatypeut.rpgle")
document = loader.load()
print(document)
print(len(document))

[Document(metadata={'source': '/content/ILEastic/unittests/mediatypeut.rpgle', 'format': 'free'}, page_content="**FREE\n\nctl-opt nomain;\n\n/include 'assert'\n/include '../headers/ileastic.rpgle'\n/include '../headers/simpleList.rpginc'\n\ndcl-ds response_extended qualified;\n  dcl-ds response likeds(il_response);\n  headerList pointer;\nend-ds;\n\ndcl-ds request likeds(il_request);\ndcl-ds responseWithHeaders likeds(response_extended);\ndcl-s optionsMethod varchar(8) inz('OPTIONS') ccsid(*utf8);\ndcl-s getMethod varchar(3) inz('GET') ccsid(*utf8);\ndcl-s marker char(8);\n\ndcl-proc test_should_return_generic_mediatype_when_no_accept_header export;\n  dcl-ds mediaType likeds(mediaType_t) inz(*likeds);\n\n  mediaType = il_mediatype_getPreferredAcceptedMediaType(request);\n\n  aEqual('*' : mediaType.type);\n  aEqual('*' : mediaType.subtype);\nend-proc;\n\ndcl-proc test_should_return_mediatype_for_single_accept_value export;\n  dcl-s headerName varchar(1024) inz('Accept');\n  dcl-s heade

###Attempt with whole GitHub Repository

Prompting the same, plus the filename.

In [ ]:
loader = RPGLoader("/content/ILEastic")
docs = loader.load()

# Cannot test all files at once (else processing error - too many calls). Selected 50 files to test.
# Passing in just the metadata causes it not to hallucinate because of too much information.
select_docs = [doc.metadata for doc in docs[:50]]
prompt = f"""Can you list just the procedure names in {select_docs} and list their associated filename of where they belong"""

response = llm.invoke(prompt)
print(response.content)



NameError: name 'llm' is not defined

###Applications: Using the LLM to suggest procedure changes

In [ ]:
prompt = f"""A customer has suggested that we should use a different CCSID.
Using these documents as context {select_docs} can you tell me which procedures to update and their full file path."""
response = llm.invoke(prompt)
print(response.content)

Based on the provided context, it appears that the CCSID used in the procedures is not explicitly mentioned. However, based on the file paths, it seems that the CCSID is likely set to 1026 (IBM-1041) in the following procedures:

- `/content/ILEastic/unittests/base64ut.rpgle` (procedure `utf8`)
- `/content/ILEastic/unittests/request.rpgle` (procedure `utf8`)
- `/content/ILEastic/unittests/smpllstut.rpgle` (procedure `lvpc`)
- `/content/ILEastic/unittests/parmlistut.rpgle` (procedure `test_noParametersPassed`, `test_twoParametersPassed`, `test_allParametersPassed`, `test_parameterOmitted`)

To update the CCSID, you would need to modify the CCSID setting in these procedures. Here are the full file paths and the procedures that need to be updated:

1. `/content/ILEastic/unittests/base64ut.rpgle` (procedure `utf8`):
   - Update the CCSID setting to the desired value.

2. `/content/ILEastic/unittests/request.rpgle` (procedure `utf8`):
   - Update the CCSID setting to the desired value.

3. 

#Chunking

In [ ]:
from langchain.text_splitter import TextSplitter
from langchain.schema import Document
from typing import Iterable, Generator, List
import re


"""
Two main functions:

Split text - Input: String, Output: List of Strings
Split documents - Input: List of Documents, Output: List of Documents

Defaults as a free format RGPLE TextSplitter

"""

class RPGLERecursiveSplitter(TextSplitter):
    def __init__(self, chunk_size=1000, chunk_overlap=20):
        super().__init__(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.current_is_free = True

    def split_text(self, text: str, is_free: bool = None) -> list[tuple[str, str]]:
      """
      User can input if the text is a free or fixed RPGLE file.
      Else, the default value is taken from the class variable self.current_is_free.
      Returns a splitting of the text dependent on format.
      """
      if is_free is None:
        is_free = self.current_is_free
      return self.split_logic_free(text) if is_free else self.split_logic_fixed(text)

    def split_documents(self, documents: Iterable[Document]) -> List[Document]:
      return list(self._split_documents(documents))

    def _split_documents(self, documents: Iterable[Document]) -> Generator[Document, None, None]:
      """
      Generator returned for memory efficiency, user calls method above
      """
      for doc in documents:
        yield from self.split_document(doc)

    def split_document(self, doc: Document) -> Generator[Document, None, None]:
      is_free = doc.metadata["format"].lower() == "free"
      chunked_docs = self.split_text(doc.page_content, is_free)
      for i, (chunk, spec) in enumerate(chunked_docs):
          new_metadata = dict(doc.metadata)
          new_metadata['chunk_number'] = i
          new_metadata['spec'] = spec
          yield Document(page_content=chunk, metadata=new_metadata)


    def split_logic_free(self, text) -> list[tuple[str, str]]:
      """
      This returns appropriate sizes chunks of text depending on whether the text is a procedure or not.
      should return a list of tuples, with the text and its specification
      """
      chunks = self.split_main(text)
      result = []
      for chunk, spec in chunks:
          if len(chunk) <= self.chunk_size:
              result.append((chunk, spec))
          else:
              result.extend(self.chunk_long_text(chunk, spec))
      return result



    def split_main(self, text):
      """
      Splits up non-procedure code using regex. Ensures dcl-xx and end-xx are not split up.
      """
      pattern = r'^(\*\*free|dcl-\w+|ctl-opt|exec sql)\b.*(?:\n[ \t].+)*'
      chunks = []
      last_end = 0
      dcl_start_pattern = re.compile(r'^dcl-(\w+)', re.IGNORECASE)

      for match in re.finditer(pattern, text, flags=re.IGNORECASE | re.MULTILINE):
          start, end = match.span()
          matched_text = text[start:end]

          # Inc text before a match (if any)
          if start > last_end:
              between_text = text[last_end:start].strip()
              if between_text:
                  chunks.append((between_text, "global"))

          if matched_text.startswith('//'):
            c_text = (matched_text, "comment")
            chunks.append(c_text)
            last_end = end
            continue

          if matched_text.startswith('**'):
              f_text = (matched_text, "format indicator")
              chunks.append(f_text)
              last_end = end
              continue

          dcl_match = dcl_start_pattern.match(matched_text.strip().lower())

          if dcl_match:
              keyword = dcl_match.group(1)
              # Look ahead in text from 'end' position to find matching 'end-keyword;'
              end_pattern = re.compile(r'^\s*end-' + re.escape(keyword) + r'\s*;', re.IGNORECASE | re.MULTILINE)
              search_pos = end

              end_match = end_pattern.search(text, search_pos)
              if end_match:

                  extended_end = end_match.end()
                  chunk = (text[start:extended_end].strip(), keyword)
                  chunks.append(chunk)
                  last_end = extended_end
                  continue


          spec = match.group(1)
          chunks.append((text[start:end].strip(), spec))
          last_end = end

      # Inc text after last match (if any)
      if last_end < len(text):
          tail_text = text[last_end:].strip()
          if tail_text:
              chunks.append((tail_text, "global"))
      return chunks


    def chunk_long_text(self, text, name):
      """
      This breaks up text that is longer than character limit, ensures overlap so does not cut off in middle of a line
      """
      chunks = []
      start = 0

      while start < len(text):
          end = start + self.chunk_size
          chunk_text = text[start:end]


          last_newline = chunk_text.rfind('\n')
          last_semicolon = chunk_text.rfind(';')
          cut_point = max(last_newline, last_semicolon)

          min_length_threshold = 0.3 #delimiter must be at least 1/3 way into the chunk
          if cut_point != -1 and cut_point > 0.3 * self.chunk_size:
              end = start + cut_point + 1


          chunk = (text[start:end], name)
          chunks.append(chunk)


          next_start = end - self.chunk_overlap
          if next_start >= len(text):
              break

          window = text[next_start:end]
          overlap_cut = max(window.rfind(';'), window.rfind('\n'))
          if overlap_cut != -1:
              start = next_start + overlap_cut + 1
          else:
              start = end



      return chunks




##Testing

In [ ]:
splitter = RPGLERecursiveSplitter()

split_text = splitter.split_text(
  """**FREE

///
// Invalid Request Example
//
// This example shows how to parse the HTTP request and return an error code to
// the caller if something is not provided or invalid. Usually the caller gets
// the HTTP code 400 (for a BAD REQUEST) with a simple error message.
//
// In this case the web service expects the caller to pass the client id as a
// query string value and it should also be a number else an error message is
// returned.
//
// Start it:
// dcl-pr SBMJOB CMD(CALL PGM(INVALIDREQ)) JOB(ILEASTIC4) JOBQ(QSYSNOMAX) ALWMLTTHD(*YES)
//
// The web service can be tested with the browser by entering the following URL:
// http://my_ibm_i:44001?client=abc
//
// @info: It requires your RPG code to be reentrant and compiled for
//        multithreading. Each client request is handled by a seperate thread.
///

ctl-opt copyright('Sitemule.com  (C), 2018');
ctl-opt decEdit('0,') datEdit(*YMD.) main(main);
ctl-opt debug(*yes) bndDir('ILEASTIC');
ctl-opt thread(*CONCURRENT);

/include ./headers/ileastic.rpgle

// -----------------------------------------------------------------------------
// Program Entry Point
// -----------------------------------------------------------------------------
dcl-proc main;

    dcl-ds config likeds(il_config);

    config.port = 44001;
    config.host = '*ANY';

    il_listen (config : %paddr(myservlet));

end-proc;

// -----------------------------------------------------------------------------
// Servlet callback implementation
// -----------------------------------------------------------------------------
dcl-proc myservlet;

    dcl-pi *n;
        request  likeds(IL_REQUEST);
        response likeds(IL_RESPONSE);
    end-pi;

    dcl-s value char(10);
    dcl-s client int(10);

    j = 123;

    // Get the client id from the query string. It should have been passed like
    // this: http://my_ibm_i:44001?client=123
    value = il_getParmStr(request : 'client');

    // Check if the client id is a valid value
    monitor;

      client = %int(value);

      // Everything is ok => return message to caller
      il_responseWrite(response : 'You passed the client id ' + %char(client));

    on-error *all;
      // Else return an error code to the caller: 400 - BAD REQUEST
      response.status = 400;
      il_responseWrite(response : 'Invalid client id passed. Client id: ' + %trim(value));
    endmon;
end-proc;""",
  True
)

for text in split_text:
  print(text)
  print("New chunk: ---------------------------")

('**FREE', 'format indicator')
New chunk: ---------------------------
('///\n// Invalid Request Example\n//\n// This example shows how to parse the HTTP request and return an error code to\n// the caller if something is not provided or invalid. Usually the caller gets\n// the HTTP code 400 (for a BAD REQUEST) with a simple error message.\n//\n// In this case the web service expects the caller to pass the client id as a\n// query string value and it should also be a number else an error message is\n// returned.\n//\n// Start it:\n// dcl-pr SBMJOB CMD(CALL PGM(INVALIDREQ)) JOB(ILEASTIC4) JOBQ(QSYSNOMAX) ALWMLTTHD(*YES)\n//\n// The web service can be tested with the browser by entering the following URL:\n// http://my_ibm_i:44001?client=abc\n//\n// @info: It requires your RPG code to be reentrant and compiled for\n//        multithreading. Each client request is handled by a seperate thread.\n///', 'global')
New chunk: ---------------------------
("ctl-opt copyright('Sitemule.com  (C), 20

#Agentic Chunking

In [ ]:
from langchain.schema import SystemMessage, HumanMessage

def agentic_chunking(file, chunk_size, llm):

    """

    Dynamically splits text into meaningful chunks using LLM.

    """
    llm = llm

    system_message = SystemMessage(content="You are an AI assistant helping to split RPGLE files into meaningful chunks for the use of LLM RAG. You can identify the difference between free and fixed format")


    human_message = HumanMessage(content=f"""
Split the following RPGLE source code into chunks.

Each chunk is a contiguous block of source code representing one logical unit, such as:

- Control specification (e.g., Ctl-Opt ...)
- Prototype (Dcl-Pr ... End-Pr)
- Procedure (Dcl-Proc ... End-Proc)
- Datastructure (Dcl-Ds ... End-Ds)
- File declaration
- Comment block (consecutive comment lines)
- Include statement

Rules:

- Do not change any character, spacing, or line breaks.
- Keep chunks exactly as they appear.
- Consecutive comment lines belong in one chunk.
- Keep original order.
- For each chunk, output the chunk **preceded by a single line** containing:

    --- [format] | [spec]

Where:

- [format] is either "free" or "fixed"
- [spec] is the chunk type (e.g., control, prototype, procedure, datastructure, filedecl, comment, include)

Example:

Input:

Dcl-Pr Example ExtPgm;
Value Char(10);
End-Pr;

Output:

--- free | prototype
Dcl-Pr Example ExtPgm;
Value Char(10);
End-Pr;

Now process the following RPGLE source:

{file}
""")



    response = llm.invoke([system_message, human_message]) # LLM returns a string

    return response.content





In [ ]:
import re
from typing import List, Tuple

def parse_chunks(text: str) -> List[Tuple[str, str, str]]:
    chunks = []
    # Split on lines starting with '---'
    parts = re.split(r'\n---\s*', '\n' + text.strip())  # Add initial \n to capture first chunk correctly
    for part in parts:
        if not part.strip():
            continue
        # First line contains "free | control" etc.
        header, *body_lines = part.splitlines()
        header = header.strip()
        m = re.match(r'(free|fixed)\s*\|\s*(\w+)', header, re.IGNORECASE)
        if not m:
            # Skip or handle error if format/spec line not found
            continue
        fmt = m.group(1).lower()
        spec = m.group(2).lower()
        chunk_text = '\n'.join(body_lines)
        if not chunk_text.strip():
            continue
        # Preserve exact chunk text including trailing blank lines if any
        chunks.append((fmt, spec, chunk_text))
    return chunks

##Testing

In [ ]:
!pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.9 MB/s  0:00:00
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain_google_genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [ ]:
import getpass
import os

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

from langchain.chat_models import init_chat_model

model = init_chat_model("gemini-2.0-flash", model_provider="google_genai")

Enter API key for Google Gemini: ··········


In [ ]:
loader = RPGLoader("/content/ILEastic/unittests/mediatypeut.rpgle")
document = loader.load()

chunks = agentic_chunking(document[0].page_content, 200, model)
print(chunks)

New chunk:  **FREE

ctl-opt nomain;

/include 'assert'
/include '../headers/ileastic.rpgle'
/include '../headers/simpleList.rpginc'

New chunk: dcl-ds response_extended qualified;
  dcl-ds response likeds(il_response);
  headerList pointer;
end-ds;

New chunk: dcl-ds request likeds(il_request);
dcl-ds responseWithHeaders likeds(response_extended);
dcl-s optionsMethod varchar(8) inz('OPTIONS') ccsid(*utf8);
dcl-s getMethod varchar(3) inz('GET') ccsid(*utf8);
dcl-s marker char(8);

New chunk: dcl-proc test_should_return_generic_mediatype_when_no_accept_header export;
  dcl-ds mediaType likeds(mediaType_t) inz(*likeds);

  mediaType = il_mediatype_getPreferredAcceptedMediaType(request);

  aEqual('*' : mediaType.type);
  aEqual('*' : mediaType.subtype);
end-proc;

New chunk: dcl-proc test_should_return_mediatype_for_single_accept_value export;
  dcl-s headerName varchar(1024) inz('Accept');
  dcl-s headerValue varchar(1024) inz('application/json');
  dcl-ds mediaType likeds(mediaType_t) i

In [ ]:
!git clone https://github.com/IBM/ibmi-company_system.git

Cloning into 'ibmi-company_system'...
remote: Enumerating objects: 668, done.
remote: Counting objects: 100% (256/256), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 668 (delta 187), reused 157 (delta 157), pack-reused 412 (from 2)
Receiving objects: 100% (668/668), 103.19 KiB | 852.00 KiB/s, done.
Resolving deltas: 100% (358/358), done.


In [ ]:
loader = RPGLoader("/content/ibmi-company_system/qrpglesrc/depts.pgm.sqlrpgle")
document = loader.load()

chunks = agentic_chunking(document[0].page_content, 200, model)
parsed_chunks = parse_chunks(chunks)
for fmt, spec, chunk in parsed_chunks:
  print(f"Format: {fmt}, Spec: {spec}\n")
  print(chunk)

Format: free, Spec: control

        Ctl-Opt DFTACTGRP(*no);
Format: free, Spec: prototype

        Dcl-Pr Employees ExtPgm;
          DepartmentNumber Char(3);
        End-Pr;
Format: free, Spec: prototype

        Dcl-Pr NewEmp ExtPgm;
          DepartmentNumber Char(3);
        End-Pr;
Format: free, Spec: comment

      //---------------------------------------------------------------*
Format: free, Spec: include

      /include 'qrpgleref/constants.rpgleinc'
Format: free, Spec: comment

      //---------------------------------------------------------------*
Format: fixed, Spec: filedecl

     Fdepts     CF   E             WorkStn Sfile(SFLDta:Rrn)
     F                                     IndDS(WkStnInd)
     F                                     InfDS(fileinfo)
Format: free, Spec: datastructure

          Dcl-S Exit Ind Inz(*Off);

          Dcl-S Rrn          Zoned(4:0) Inz;

          Dcl-DS WkStnInd;
            ProcessSCF     Ind        Pos(21);
            ReprintScf     In

In [ ]:
parsed_chunks = parse_chunks(chunks)
for fmt, spec, chunk in parsed_chunks:
  print(f"Format: {fmt}, Spec: {spec}\n")
  print(chunk)

Format: free, Spec: control

        Ctl-Opt DFTACTGRP(*no);
Format: free, Spec: prototype

        Dcl-Pr Employees ExtPgm;
          DepartmentNumber Char(3);
        End-Pr;
Format: free, Spec: prototype

        Dcl-Pr NewEmp ExtPgm;
          DepartmentNumber Char(3);
        End-Pr;
Format: free, Spec: comment

      //---------------------------------------------------------------*
Format: free, Spec: include

      /include 'qrpgleref/constants.rpgleinc'
Format: free, Spec: comment

      //---------------------------------------------------------------*
Format: fixed, Spec: filedecl

     Fdepts     CF   E             WorkStn Sfile(SFLDta:Rrn)
     F                                     IndDS(WkStnInd)
     F                                     InfDS(fileinfo)
Format: free, Spec: datastructure

          Dcl-S Exit Ind Inz(*Off);

          Dcl-S Rrn          Zoned(4:0) Inz;

          Dcl-DS WkStnInd;
            ProcessSCF     Ind        Pos(21);
            ReprintScf     In